## Importing packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, TimeSeriesSplit
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    mean_absolute_error,
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.compose import TransformedTargetRegressor

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.model_selection import KFold

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt
import joblib
from datetime import datetime

import seaborn as sns
import warnings

import wandb
from wandb.integration.xgboost import WandbCallback
import numpy as np
import xgboost as xgb
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)
warnings.filterwarnings('ignore')

## Data Loading & Preparation

In [2]:
df = pd.read_csv("../data_model/db_additional_fe.csv")
df_test = pd.read_csv("../data_model/db_collected_fe.csv")

def sanitize_column_names(df):
    """Replace special JSON characters in column names with underscores."""
    import re
    sanitized = {}
    for col in df.columns:
        sanitized_col = re.sub(r'[\[\](){},\"\'\\\/\s]', '_', str(col))
        sanitized_col = re.sub(r'_+', '_', sanitized_col)
        sanitized_col = sanitized_col.strip('_')
        sanitized[col] = sanitized_col
    df = df.rename(columns=sanitized)
    return df

df = sanitize_column_names(df)
df_test = sanitize_column_names(df_test)

print(f"Data loaded: Train {df.shape}, Test {df_test.shape}")

Data loaded: Train (97927, 1556), Test (85419, 2716)


In [3]:
def align_dataframes(df1, df2, fill_value=0):
    """
    Aligns two DataFrames to have the exact same columns (union of both).
    Missing columns are added and filled with `fill_value`.
    """
    # 1. Get the union of all column names
    all_cols = df1.columns.union(df2.columns)
    
    # 2. Reindex both DataFrames to this full set of columns
    #    'fill_value' handles the new missing columns
    df1_aligned = df1.reindex(columns=all_cols, fill_value=fill_value)
    df2_aligned = df2.reindex(columns=all_cols, fill_value=fill_value)

    df1_aligned.sort_values(by='snapshot_ts', ascending=False, inplace=True)
    df2_aligned.sort_values(by='snapshot_ts', ascending=False, inplace=True)

    df1_aligned.drop_duplicates(subset='market_id', inplace=True)
    df2_aligned.drop_duplicates(subset='market_id', inplace=True)
    
    # Optional: Ensure columns are in the exact same order (reindex does this, but good to be sure)
    # The .union() usually sorts them, so they will match.
    
    return df1_aligned, df2_aligned

# --- Usage Example ---
df, df_test = align_dataframes(df, df_test, fill_value=0)

print(f"Aligned shapes: Train {df.shape}, Test {df_test.shape}")

Aligned shapes: Train (97653, 2720), Test (3905, 2720)


In [4]:
y_train_clf = (df['volume_total'] > 10_000).astype(int)
X_train_clf = df.drop(columns=['volume_total'])

y_test_clf = (df_test['volume_total'] > 10_000).astype(int)
X_test_clf = df_test.drop(columns=['volume_total'])

print(f"Classification data - Train: {X_train_clf.shape}, Test: {X_test_clf.shape}")
print(f"Train distribution: {(y_train_clf==0).sum()} negative ({(y_train_clf==0).sum()/len(y_train_clf):.1%}), "
      f"{(y_train_clf==1).sum()} positive ({(y_train_clf==1).sum()/len(y_train_clf):.1%})")
print(f"Test distribution: {(y_test_clf==0).sum()} negative ({(y_test_clf==0).sum()/len(y_test_clf):.1%}), "
      f"{(y_test_clf==1).sum()} positive ({(y_test_clf==1).sum()/len(y_test_clf):.1%})")


X_train_clf = X_train_clf.fillna(0)
X_test_clf = X_test_clf.fillna(0)

X_train_clf = X_train_clf.filter(regex='emb')
X_test_clf = X_test_clf.filter(regex='emb')

Classification data - Train: (97653, 2719), Test: (3905, 2719)
Train distribution: 48893 negative (50.1%), 48760 positive (49.9%)
Test distribution: 3502 negative (89.7%), 403 positive (10.3%)


In [5]:
pos_count = (y_train_clf == 1).sum()
neg_count = (y_train_clf == 0).sum()
total_count = len(y_train_clf)

current_pos_ratio = pos_count / total_count
current_neg_ratio = neg_count / total_count

target_pos_ratio = 0.05 #from test set
target_neg_ratio = 0.95 #from test set

weight_pos = target_pos_ratio / current_pos_ratio
weight_neg = target_neg_ratio / current_neg_ratio

sample_weights = np.where(y_train_clf == 1, weight_pos, weight_neg)

print(f"Train distribution: {neg_count} negative ({current_neg_ratio:.1%}), {pos_count} positive ({current_pos_ratio:.1%})")
print(f"Target distribution: 95% negative, 5% positive")
print(f"Weight for class 0 (negative): {weight_neg:.3f}")
print(f"Weight for class 1 (positive): {weight_pos:.3f}")
print(f"Sample weights shape: {sample_weights.shape}")
print(f"Sample weights ready")

Train distribution: 48893 negative (50.1%), 48760 positive (49.9%)
Target distribution: 95% negative, 5% positive
Weight for class 0 (negative): 1.897
Weight for class 1 (positive): 0.100
Sample weights shape: (97653,)
Sample weights ready


In [6]:
RANDOM_STATE = 2026
N_JOBS = -1

In [ ]:
best_xgb_params = {
    "n_estimators": 464,
    "max_depth": 6,
    "learning_rate": 0.0916873039922197,
    "subsample": 0.9122237063028887,
    "colsample_bytree": 0.9136901715271788,
    "reg_alpha": 0.9308494604559847,
    "reg_lambda": 4.215531420238764,
    "min_child_weight": 5,
    "early_stopping_rounds": 13,
    "eta": 0.4972898002627886,
}

print("Best XGBoost params from Optuna:")
for k, v in best_xgb_params.items():
    print(f"  {k}: {v}")

# 2. Prepare DMatrices
dtrain = xgb.DMatrix(X_train_clf, label=y_train_clf, weight=sample_weights)
dtest = xgb.DMatrix(X_test_clf, label=y_test_clf)

# XGBoost param dict (use boosters' names, not sklearn wrapper names)
xgb_params = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "seed": RANDOM_STATE,
    "nthread": N_JOBS,
}
xgb_params.update(best_xgb_params)

# 3. Start a new W&B run
wandb_run = wandb.init(
    entity="piotr-wilma-uniwersytet-ekonomiczny-w-poznaniu",
    project="Uczenie Maszynowe",
    name="xgb_classifier_final",
    config={
        "model_type": "xgb_weighted",
        "optuna_best_params": best_xgb_params,
        "n_train_samples": int(X_train_clf.shape[0]),
        "n_features": int(X_train_clf.shape[1]),
    },
)

# 4. Train with WandbCallback
evals = [(dtrain, "train"), (dtest, "valid")]
num_boost_round = best_xgb_params.get(
    "n_estimators", best_xgb_params.get("n_rounds", 400)
)

bst = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    evals=evals,
    callbacks=[WandbCallback()],  # logs metrics and parameters to W&B
)



Best XGBoost params from Optuna:
  n_estimators: 464
  max_depth: 6
  learning_rate: 0.0916873039922197
  subsample: 0.9122237063028887
  colsample_bytree: 0.9136901715271788
  reg_alpha: 0.9308494604559847
  reg_lambda: 4.215531420238764
  min_child_weight: 5
  early_stopping_rounds: 13
  eta: 0.4972898002627886


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/an0da/.netrc.
wandb: Currently logged in as: an0da (an0da-pozna-university-of-economics-and-business) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[0]	train-aucpr:0.27357	valid-aucpr:0.12249
[1]	train-aucpr:0.31420	valid-aucpr:0.11697
[2]	train-aucpr:0.33120	valid-aucpr:0.11552
[3]	train-aucpr:0.34869	valid-aucpr:0.10868
[4]	train-aucpr:0.36408	valid-aucpr:0.10688
[5]	train-aucpr:0.37375	valid-aucpr:0.10633
[6]	train-aucpr:0.38234	valid-aucpr:0.10712
[7]	train-aucpr:0.38912	valid-aucpr:0.10945
[8]	train-aucpr:0.39325	valid-aucpr:0.10770
[9]	train-aucpr:0.40064	valid-aucpr:0.10582
[10]	train-aucpr:0.40489	valid-aucpr:0.10457
[11]	train-aucpr:0.40954	valid-aucpr:0.10480
[12]	train-aucpr:0.41520	valid-aucpr:0.10425
[13]	train-aucpr:0.41871	valid-aucpr:0.10400
[14]	train-aucpr:0.42344	valid-aucpr:0.10257
[15]	train-aucpr:0.42771	valid-aucpr:0.10122
[16]	train-aucpr:0.43171	valid-aucpr:0.10010
[17]	train-aucpr:0.43648	valid-aucpr:0.09789
[18]	train-aucpr:0.43981	valid-aucpr:0.09674
[19]	train-aucpr:0.44300	valid-aucpr:0.09594
[20]	train-aucpr:0.44682	valid-aucpr:0.09507
[21]	train-aucpr:0.44985	valid-aucpr:0.09493
[22]	train-aucpr:0.4

AttributeError: 'Booster' object has no attribute 'predict_proba'

In [9]:
dtest = xgb.DMatrix(X_test_clf, label=y_test_clf)

# 5. Evaluate on test set and log metrics
y_pred_proba = bst.predict(dtest)
precisions, recalls, thresholds = precision_recall_curve(y_test_clf, y_pred_proba)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx] if len(thresholds) > best_threshold_idx else 0.5


y_proba_final = bst.predict(dtest)

test_pr_auc_final = average_precision_score(y_test_clf, y_proba_final)
test_roc_auc_final = roc_auc_score(y_test_clf, y_proba_final)

best_threshold_final = best_threshold
y_pred_final = (y_proba_final >= best_threshold_final).astype(int)

test_f1_final = f1_score(y_test_clf, y_pred_final)
test_precision_final = precision_score(y_test_clf, y_pred_final, zero_division=0)
test_recall_final = recall_score(y_test_clf, y_pred_final)

wandb.summary["test_pr_auc"] = float(test_pr_auc_final)
wandb.summary["test_roc_auc"] = float(test_roc_auc_final)
wandb.summary["test_f1"] = float(test_f1_final)
wandb.summary["test_precision"] = float(test_precision_final)
wandb.summary["test_recall"] = float(test_recall_final)
wandb.summary["decision_threshold"] = float(best_threshold_final)

print("Final XGB metrics (test):")
print(f"  PR-AUC:     {test_pr_auc_final:.4f}")
print(f"  ROC-AUC:    {test_roc_auc_final:.4f}")
print(f"  F1:         {test_f1_final:.4f}")
print(f"  Precision:  {test_precision_final:.4f}")
print(f"  Recall:     {test_recall_final:.4f}")
print(f"  Threshold:  {best_threshold_final:.4f}")

# 6. Log predictions artifact for submission/inspection
submission_df = pd.DataFrame(
    {
        "market_id": df_test["market_id"].values
        if "market_id" in df_test.columns
        else np.arange(len(y_proba_final)),
        "y_proba": y_proba_final,
        "y_pred": y_pred_final,
    }
)

submission_path = "xgb_optuna_best_predictions.csv"
submission_df.to_csv(submission_path, index=False)

artifact = wandb.Artifact(
    name="xgb_optuna_best_predictions",
    type="predictions",
    description="Final XGB model predictions using best Optuna params",
)
artifact.add_file(submission_path)
wandb_run.log_artifact(artifact)

# 7. Finish W&B run
wandb.finish()


Final XGB metrics (test):
  PR-AUC:     0.0992
  ROC-AUC:    0.4503
  F1:         0.1878
  Precision:  0.1038
  Recall:     0.9801
  Threshold:  0.0029


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train-aucpr,▁▁▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████
valid-aucpr,█▆▃▂▂▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄
decision_threshold,0.00295
epoch,463
test_f1,0.18778
test_pr_auc,0.09922
test_precision,0.10384
test_recall,0.98015
test_roc_auc,0.45029


In [10]:
xgb_params

{'objective': 'binary:logistic',
 'tree_method': 'hist',
 'eval_metric': 'aucpr',
 'seed': 2026,
 'nthread': -1,
 'n_estimators': 464,
 'max_depth': 6,
 'learning_rate': 0.0916873039922197,
 'subsample': 0.9122237063028887,
 'colsample_bytree': 0.9136901715271788,
 'reg_alpha': 0.9308494604559847,
 'reg_lambda': 4.215531420238764,
 'min_child_weight': 5,
 'early_stopping_rounds': 13,
 'eta': 0.4972898002627886}